# 04. Pesquisa local: Hill Climbing e Simulated Annealing

Este notebook trata a pesquisa local do projeto (`src/local_search.py`). Mostra a vizinhança, compara duas variantes de Hill Climbing com o Simulated Annealing, mede a
sensibilidade do SA aos parâmetros e o efeito da solução inicial.

## O problema

Uma plataforma liga pequenos produtores do distrito de Braga aos clientes através de um centro de
recolha com dois veículos. Cada produto é recolhido no produtor, descarregado no armazém e, mais tarde,
carregado e entregue ao cliente. O agente tem de servir todas as encomendas com o menor número de km,
respeitando as janelas de recolha e de entrega (que limitam a hora a que o serviço começa), a
capacidade dos veículos, 10 min de serviço por paragem, 15 min para descarregar e 15 para carregar no
armazém, e no máximo 24 h de permanência lá.

O modelo (`src/model.py`) junta as linhas de encomenda em pedidos. Um pedido de recolha (P1, P2, ...)
agrupa as linhas de um produtor cujas janelas se intersetam; um pedido de entrega (D1, D2, ...) é uma
encomenda e depende das recolhas que trazem os seus produtos. Cada veículo faz uma sequência de viagens
que saem do centro e voltam lá, cada uma só com recolhas ou só com entregas, e nenhuma passa a
meia-noite. Os veículos saem *just in time*, para chegar à primeira paragem quando a janela abre, e um
veículo pode entregar o que outro recolheu, desde que já esteja descarregado. O tiny tem 18 pedidos e
o small 37.

`evaluate(problem, solution)` simula uma solução e devolve os km e as violações; `check_solution` volta
a verificá-la linha a linha, diretamente sobre os CSV.

In [ ]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from collections import Counter

from src.model import load_problem, evaluate, check_solution, plot_routes, plot_gantt, plot_histories
from src.informed import greedy, h_mst
from src.local_search import hill_climbing, simulated_annealing, neighbours
from src.csp import solve

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
ROOT = Path.cwd().parent
REPORTS = ROOT / "reports"
tiny = load_problem(ROOT / "data" / "tiny")
small = load_problem(ROOT / "data" / "small")
PROBLEMS = {"tiny": tiny, "small": small}


def load_solution(filename):
    """Lê uma solução gravada em reports/ como {veículo: [viagens]}."""
    return {int(vehicle): trips for vehicle, trips in json.loads((REPORTS / filename).read_text()).items()}


def print_routes(solution):
    for vehicle, trips in solution.items():
        print(f"Veículo {vehicle}:", " | ".join("W→" + "→".join(trip) + "→W" for trip in trips))


best = json.loads((REPORTS / "best.json").read_text())
starts = {name: greedy(problem, h_mst).solution for name, problem in PROBLEMS.items()}

In [ ]:
# os 18 pedidos do tiny depois do agrupamento
rows = [dict(id=request_id, tipo=request.kind, local=request.loc,
             janela=f"{tiny.clock(request.tw[0])} – {tiny.clock(request.tw[1])}",
             kg=request.weight, linhas=len(request.items), depende_de=", ".join(request.deps))
        for request_id, request in tiny.requests.items()]
pd.DataFrame(rows).set_index("id").T

## 1. Representação, objetivo e algoritmos

A pesquisa local trabalha sobre soluções completas, dicionários `{veículo: [viagens]}` em que cada
viagem é uma lista de pedidos. Para comparar soluções, incluindo as inviáveis, usa o custo penalizado
de `evaluate`:


`km + 100·(atraso + excesso de carga + excesso de permanência) + 10 000·(erros estruturais)`


Os erros estruturais são pedidos em falta ou repetidos, bloqueios entre veículos, viagens mistas e
viagens que passam a meia-noite. Numa solução viável, o custo penalizado é igual aos km.

O Hill Climbing *steepest* avalia a vizinhança toda, fica com o melhor vizinho viável e pára num ótimo
local. A variante *first improvement* percorre a vizinhança por ordem aleatória e aceita o primeiro
vizinho viável que melhora. O Simulated Annealing escolhe um vizinho ao acaso e aceita-o com
probabilidade exp(−Δ/T), com arrefecimento geométrico (T₀ = 30, α = 0,9998, 30 000 iterações). Pode
passar por soluções inviáveis, mas devolve a melhor solução viável que encontrou. Os três partem da
solução do Greedy com h3.

## 2. Vizinhança

A vizinhança tem três movimentos, e nenhum mistura recolhas e entregas na mesma viagem: *relocate* tira
um pedido e põe-no noutra posição, noutra viagem, noutro veículo ou numa viagem nova; *swap* troca dois
pedidos do mesmo tipo; *2-opt* inverte um segmento dentro de uma viagem.

A célula seguinte conta os vizinhos da solução Greedy de cada dataset, por tipo de movimento.

In [ ]:
for name, problem in PROBLEMS.items():
    moves = list(neighbours(problem, starts[name]))
    move_counts = Counter(move[0] for move, _ in moves)
    feasible = sum(evaluate(problem, neighbour).feasible for _, neighbour in moves)
    print(name, dict(move_counts), "| total:", len(moves), "| viáveis:", feasible)

## 3. Hill Climbing a partir do Greedy (h3)

In [ ]:
hill_climbs = {}
for name, problem in PROBLEMS.items():
    hill_climbs[name, "steepest"] = hill_climbing(problem, starts[name], "steepest")
    for seed in range(5):
        hill_climbs[name, f"first-{seed}"] = hill_climbing(problem, starts[name], "first", seed=seed)
pd.DataFrame([dict(dataset=name, variante=variant, **result.summary())
              for (name, variant), result in hill_climbs.items()])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, PROBLEMS):
    plot_histories({variant: dict(current=result.history) for (dataset, variant), result in hill_climbs.items()
                    if dataset == name}, f"{name}: Hill Climbing", ax=ax)
    ax.axhline(best[name]["km"], color="grey", ls=":", label="ótimo")
    ax.legend(fontsize=7)

In [ ]:
for name in PROBLEMS:
    print(name, "movimentos aceites (steepest):", [move[0] for move in hill_climbs[name, "steepest"].moves])

O *steepest* escolhe sempre o melhor vizinho. Faz poucas iterações, e cada uma avalia a vizinhança
toda. O *first improvement* aceita o primeiro vizinho que melhora, por isso cada iteração custa menos e
são precisas mais. Como a ordem é aleatória, cada semente pode acabar num ótimo local diferente.

## 4. Simulated Annealing

In [ ]:
annealing = {name: simulated_annealing(problem, starts[name], t0=30, alpha=0.9998, iters=30_000, seed=0)
             for name, problem in PROBLEMS.items()}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, name in zip(axes, PROBLEMS):
    plot_histories({"atual": dict(current=annealing[name].history)}, f"{name}: SA (T0=30, α=0.9998)", ax=ax,
                   cap=best[name]["km"] * 1.6)
    best_so_far = pd.Series(annealing[name].best_history).replace(float("inf"), None)
    ax.plot(best_so_far.index, best_so_far.values, color="black", lw=1.5, label="melhor viável")
    ax.legend()
pd.DataFrame([dict(dataset=name, **result.summary()) for name, result in annealing.items()])

### Sensibilidade a T₀ e α

No small, com 3 sementes e 15 000 iterações por corrida. T₀ controla quanto se aceitam pioras no início
e α a velocidade do arrefecimento: com α = 0,9995, ao fim de 15 000 iterações T ≈ T₀ · 0,0006.

In [ ]:
rows = []
for t0 in (5, 30, 100):
    for alpha in (0.999, 0.9995, 0.9998):
        for seed in range(3):
            result = simulated_annealing(small, starts["small"], t0=t0, alpha=alpha, iters=15_000, seed=seed)
            rows.append(dict(T0=t0, alpha=alpha, seed=seed, km=result.cost, viavel=result.feasible,
                             segundos=result.seconds))
sensitivity = pd.DataFrame(rows)
sensitivity.pivot_table(index="T0", columns="alpha", values="km", aggfunc=["mean", "min"]).round(1)

## 5. Efeito da solução inicial (small)

In [ ]:
csp_start = solve(small, global_fc=True).solution
# solução trivial: um pedido por viagem, por ordem de janela, alternando os veículos
trivial = {vehicle.vid: [] for vehicle in small.vehicles}
for i, request_id in enumerate(sorted(small.requests, key=lambda r: small.requests[r].tw[0])):
    trivial[small.vehicles[i % 2].vid].append([request_id])

rows = []
for label, start in [("Greedy h3", starts["small"]), ("CSP (MAC+MRV+FC global)", csp_start),
                     ("trivial (1 pedido por viagem)", trivial)]:
    initial = evaluate(small, start)
    climb = hill_climbing(small, start, "steepest") if initial.feasible else None
    anneal = simulated_annealing(small, start, t0=30, alpha=0.9998, iters=30_000, seed=0)
    rows.append(dict(inicio=label, km_inicial=round(initial.km, 2), inicial_viavel=initial.feasible,
                     HC_steepest=None if climb is None else round(climb.cost, 2),
                     SA=round(anneal.cost, 2) if anneal.feasible else None))
pd.DataFrame(rows)

A solução trivial é inviável: tem atrasos e passa a permanência máxima. O Hill Climbing só se move entre
soluções viáveis, por isso não pode partir dela. O SA caminha pelo custo penalizado e consegue
repará-la até chegar a uma solução viável. A solução inicial também muda o ótimo local a que o HC chega:
799,35 km a partir do Greedy e 785,50 km a partir do CSP.

## 6. Melhor solução encontrada no small

In [ ]:
candidates = [(result.cost, f"HC {variant}", result.solution) for (name, variant), result in hill_climbs.items()
              if name == "small" and result.feasible]
candidates.append((annealing["small"].cost, "SA", annealing["small"].solution))
_, best_label, best_solution = min(candidates, key=lambda candidate: candidate[0])
best_eval = evaluate(small, best_solution)
print(best_label, best_eval.summary(), "| verificador:", check_solution(small, best_eval).valid)
plot_routes(small, best_solution, f"small: {best_label}, {best_eval.km:.2f} km");
plot_gantt(small, best_eval);

## 7. Conclusões

No tiny, o HC e o SA chegam ambos ao ótimo provado pelo A\*: o HC em décimas de segundo, o SA em cerca
de 1,2 s.

No small, a referência é o ótimo provado pelo A\* por viagens, a linha pontilhada nos gráficos. O HC
*steepest* e algumas sementes do *first improvement* ficam presos num ótimo local acima dele. O SA
depende muito de T₀, de α e da semente; explora mais, e mesmo assim nem sempre bate o HC. Tem a
vantagem de reparar soluções inviáveis, enquanto o HC precisa de uma solução viável para começar.

O ponto de partida conta. Neste problema, a combinação mais eficiente é uma boa construção inicial
(Greedy ou CSP) seguida de pesquisa local.